# Train Axiom 500M on 56K pairs

5 sources: semiconductor + distilled + teacher R1 + teacher R2 + conversational.
Resumable. Atomic Drive saves. Each step PASS/FAIL gated.

In [ ]:
# Step 1: Mount Drive + verify space
from google.colab import drive
drive.mount('/content/drive')
import shutil
usage = shutil.disk_usage('/content/drive/MyDrive')
free_gb = usage.free / 1e9
assert free_gb > 5, f'FAIL: {free_gb:.1f}GB free, need >5GB'
print(f'PASS: Drive mounted, {free_gb:.1f}GB free')

In [ ]:
# Step 2: Verify Drive write
import torch, os
path = '/content/drive/MyDrive/axiom_drive_test.pt'
torch.save({'test': True}, path)
loaded = torch.load(path, map_location='cpu')
assert loaded['test'] == True
os.remove(path)
print('PASS: Drive write verified')

In [ ]:
# Step 3: Clone + install
!git clone https://github.com/MetaCortex-Dynamics/Axiom.git
%cd Axiom
!pip install -q torch tokenizers

In [ ]:
# Step 4: Verify corpus (5 sources, >= 50K)
import json, os
paths = [
    'corpus/axiom/pairs.json',
    'corpus/distilled/self_distilled_pairs.json',
    'corpus/teacher/teacher_pairs.json',
    'corpus/teacher/teacher_pairs_r2.json',
    'corpus/conversational_pairs.json',
]
total = 0
for p in paths:
    assert os.path.exists(p), f'FAIL: {p} missing'
    with open(p) as f:
        n = len(json.load(f))
    print(f'  {p}: {n}')
    total += n
assert total >= 50000, f'FAIL: only {total} pairs'
print(f'PASS: {total} pairs')

In [ ]:
# Step 5: Verify GPU
import torch
assert torch.cuda.is_available(), 'FAIL: no CUDA'
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'PASS: {name}, {vram:.0f}GB')

In [ ]:
# Step 6: Verify script fixes
src = open('scripts/train_500m_v2.py').read()
checks = [
    ('os.makedirs', 'makedirs' in src),
    ('full checkpoint', '"optimizer"' in src and '"scheduler"' in src),
    ('Drive space', 'disk_usage' in src),
    ('resume', 'load_checkpoint' in src),
    ('atomic rename', 'os.replace' in src),
    ('Drive log', 'DRIVE_LOG' in src),
    ('tmp save', '.tmp.pt' in src),
    ('[SAVE] log', '[SAVE]' in src),
    ('5 sources', 'teacher_pairs_r2' in src and 'conversational' in src),
]
all_pass = True
for name, ok in checks:
    if not ok: all_pass = False
    print(f'  [{"PASS" if ok else "FAIL"}] {name}')
assert all_pass, 'FAIL: script missing fixes'
print('PASS: all verified')

In [ ]:
# Step 7: Train
!python -u scripts/train_500m_v2.py 2>&1 | tee /content/drive/MyDrive/axiom_500m_56k_train.log

In [ ]:
# Step 8: Verify checkpoint
import torch
path = '/content/drive/MyDrive/axiom_500m_56k_checkpoint.pt'
ckpt = torch.load(path, map_location='cpu')
print(f'PASS: epoch={ckpt["epoch"]}, best={ckpt["best"]:.4f}, keys={len(ckpt["model"])}')

In [ ]:
# Step 9: Download
from google.colab import files
files.download('/content/drive/MyDrive/axiom_500m_decoder_final.pt')